# GNLSE 3+1D Solver Tutorial

A comprehensive introduction to nonlinear pulse propagation in optical fibers using our JAX-based Generalized Nonlinear Schrödinger Equation (GNLSE) solver.

## Table of Contents

0. **Preamble** — Physics background, solver overview, imports, helper factory
1. **Linear Free-Space Propagation** — Diffraction in 1D, 2D, 3D (no dispersion, no nonlinearity)
2. **Linear Dispersion (1+1D)** — Pulse broadening from group velocity dispersion
3. **Nonlinear, No Dispersion (1+1D)** — Self-phase modulation and spectral broadening
4. **Nonlinear + Dispersion (1+1D)** — Pulse distortion and soliton formation
5. **Linear Fiber Propagation (2+1D)** — Waveguide modes, GRIN and step-index fibers
6. **Nonlinear Fiber Propagation (2+1D)** — Kerr nonlinearity in multimode fibers
7. **Full 3+1D Nonlinear Pulse Propagation** — All effects combined
8. **Gradient Descent for Soliton Profile** — Optimizing a pulse shape via windowed backpropagation

## Physics Background

The **Generalized Nonlinear Schrödinger Equation (GNLSE)** describes the evolution of the slowly varying electric field envelope $A(x, y, t, z)$ along the propagation direction $z$:

$$\frac{\partial \tilde{A}}{\partial z} = i\left[\sqrt{\beta_{\text{eff}}^2(\omega) - k_\perp^2} - \beta_0 - \beta_1 \omega - \tfrac{1}{2}\beta_2 \omega^2\right]\tilde{A} + i\gamma |A|^2 A + i\gamma f_R (h_R \otimes |A|^2)A + \cdots$$

where $\tilde{A}$ is the field in $(k_x, k_y, \omega)$ space.

### Physical Effects

| Term | Physics | Parameters |
|------|---------|------------|
| $\sqrt{\beta_{\text{eff}}^2 - k_\perp^2}$ | **Diffraction** — transverse spreading | $n(x,y)$, $\lambda_0$ |
| $-\tfrac{1}{2}\beta_2 \omega^2$ | **Dispersion** — frequency-dependent group velocity | $\beta_2$ (GVD) |
| $i\gamma \vert A \vert^2 A$ | **Kerr nonlinearity** — intensity-dependent refractive index | $n_2$, $\gamma = n_2 \omega_0 / c$ |
| $f_R (h_R \otimes \vert A \vert^2)$ | **Raman scattering** — delayed nonlinear response | $f_R$, $\tau_1$, $\tau_2$ |
| Self-steepening | **Shock formation** at pulse edges | `sw=1` |
| Gain | **Saturable amplification** | `gain_coeff`, `gain_fwhm` |

### Key Length Scales

- **Rayleigh length**: $z_R = \pi w_0^2 / \lambda_0$ — distance over which a beam doubles its area
- **Dispersion length**: $L_D = T_0^2 / |\beta_2|$ — distance for pulse to broaden by $\sqrt{2}$
- **Nonlinear length**: $L_{NL} = 1/(\gamma P_0)$ — distance for $\pi$ phase shift via SPM
- **Critical power**: $P_{cr} = \lambda_0^2 / (2\pi n_2)$ — self-focusing threshold

## Solver Overview

The solver uses the **split-step Fourier method**:

1. **Half-step linear** (spectral domain): apply $D_{\text{half}} = e^{i D \Delta z/2}$ where $D$ contains diffraction + dispersion
2. **Full-step nonlinear** (time domain): Kerr half-steps bracket a Heun integrator for Raman/self-steepening
3. **Half-step linear**: second half of spectral propagation

### Field Representation

- Fields are 3D complex arrays `(Nx, Ny, Nt)` representing the spatial and temporal envelope
- Internally propagated in k-space as `field_kwo` $(k_x, k_y, \omega)$ via FFT
- Materialized to $(x, y, t)$ via `_materialize_xyt()` for saves/events

### Singleton Axis Safety

When `Nx=1` or `Ny=1`, PML is disabled on that axis, enabling 1D and 2D simulations with the same 3D solver.

### Memory Management

- **Checkpointing**: `"segments"` (default), `"tree"`, or `"none"` for memory/speed tradeoff
- **Windowed propagation**: breaks z into independent JIT windows for bounded GPU memory during gradient computation
- **`windowed_grad()`**: per-window VJP in reverse for memory-efficient backpropagation

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"
# Leave headroom for cuFFT compilation scratch — 0.90 can cause OOM at high Nt
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.70"

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

jax.config.update("jax_enable_x64", True)

from gnlse_solver_noisy import (
    make_windowed_context_noisy,
    windowed_forward_noisy,
    windowed_grad_noisy,
    propagate_windowed,
    _materialize_xyt, _make_mesh_for_time_axis, _resolve_precision,
    make_args,
)

# Noiseless shims — route through gnlse_solver_noisy with eps=None, noise_sigma=0
def make_windowed_context(args, Nt, *, precision=None):
    return make_windowed_context_noisy(args, Nt, precision=precision, loss_coeff=0.0)

def windowed_forward(args, A0, *, n_windows=10, ctx=None, save_as_fp32=False):
    return windowed_forward_noisy(args, A0, None, 0.0,
                                  n_windows=n_windows, ctx=ctx,
                                  save_as_fp32=save_as_fp32)

def windowed_grad(loss_fn, args, A0, *, n_windows=10, ctx=None):
    return windowed_grad_noisy(loss_fn, args, A0, None, 0.0,
                                n_windows=n_windows, ctx=ctx)

from gnlse_source_prototype import (
    solve_modes, make_source_from_files, norm_scale_field_weights,
    combine_spatial_temporal, cw_temp_profile_freq, gaussian_pulse_profile_freq,
)
from gnlse_medium import make_space, make_polynomial_n, make_supergauss_index
from gnlse_visualizations_prototype import (
    make_xy_t_animation, make_xy_z_animation,
    make_transverse_vs_z_vs_I_plot,
    make_1d_z_animation, make_temporal_vs_z_plot,
    plot_modes_gallery,
)

# Physical constants
c0 = 2.998e8          # speed of light [m/s]
lambda0 = 1064e-9     # central wavelength [m]
n_core = 1.453        # core refractive index
NA = 0.1              # numerical aperture
n_clad = np.sqrt(n_core**2 - NA**2)
r_core_ref = 25e-6    # reference core radius [m]
n2 = 2.76e-20         # Kerr coefficient [m^2/W]
beta0 = n_core * 2 * np.pi / lambda0
n_g = 1.468           # group index
beta1 = n_g / c0      # inverse group velocity [s/m]
beta2 = 22e-27         # GVD [s^2/m] (normal dispersion)
omega0 = 2 * np.pi * c0 / lambda0
gamma = n2 * omega0 / c0  # bulk nonlinear coefficient [m/W]

# Effective mode area for 1D temporal simulations (Nx=Ny=1).
# The solver uses bulk gamma, so the field must be in sqrt(intensity [W/m^2]),
# NOT sqrt(power [W]). For 1D sections we convert: I0 = P0 / A_eff.
A_eff = np.pi * r_core_ref**2  # ~ 1.96e-9 m^2

# Raman parameters
t1_raman = 12.2e-15   # [s]
t2_raman = 32e-15     # [s]
fr_raman = 0.18       # Raman fraction

print(f"lambda0 = {lambda0*1e9:.1f} nm")
print(f"n_core = {n_core}, n_clad = {n_clad:.4f}")
print(f"gamma = {gamma:.4e} m/W (bulk)")
print(f"A_eff = {A_eff:.3e} m^2  =>  gamma_fiber = gamma/A_eff = {gamma/A_eff:.4f} 1/(W*m)")
print(f"beta2 = {beta2:.2e} s^2/m")
print(f"JAX devices: {jax.devices()}")

In [ ]:
# Bind the tutorial's physical constants into make_args so call sites
# stay concise.  The generic make_args lives in gnlse_solver_new.
from functools import partial

make_args = partial(
    make_args,
    lambda0=lambda0,
    n_ref=n_core,
    beta0_val=beta0,
    beta1_val=beta1,
    t1=t1_raman,
    t2=t2_raman,
)


---
# Section 1: Linear Free-Space Propagation (No Dispersion)

**Diffraction**: A beam spreads transversely in free space due to the wave nature of light. With no nonlinearity ($n_2 = 0$) and no dispersion ($\beta_2 = 0$), the solver reduces to a **beam propagation method (BPM)**.

For a Gaussian beam with waist $w_0$, the beam width evolves as:
$$w(z) = w_0 \sqrt{1 + \left(\frac{z}{z_R}\right)^2}, \quad z_R = \frac{\pi w_0^2}{\lambda_0}$$

## 1a: 1+1D Free-Space Diffraction (Nx>1, Ny=1, Nt=1)

In [ ]:
# --- Tunable parameters ---
Nx_1a = 256
Lx_1a = 400e-6      # 400 um transverse window
Lz_1a = 5e-3        # 5 mm propagation
w0_1a = 20e-6        # 20 um beam waist

# Derived
z_R_1a = np.pi * n_core * w0_1a**2 / lambda0  # in-medium Rayleigh length
deltaZ_1a = z_R_1a / 1000
print(f"Rayleigh length z_R = {z_R_1a*1e3:.3f} mm")
print(f"deltaZ = z_R/10 = {deltaZ_1a*1e3:.3f} mm")
print(f"Lz / z_R = {Lz_1a / z_R_1a:.1f}")
print(f"Total steps ~ {int(round(Lz_1a / deltaZ_1a))}")

In [ ]:
# Build uniform index, Gaussian beam, CW temporal
dx_1a = Lx_1a / Nx_1a
x_1a = np.linspace(-Lx_1a/2, Lx_1a/2, Nx_1a)

# Gaussian beam: exp(-x^2 / w0^2)
E_x_1a = np.exp(-x_1a**2 / w0_1a**2).astype(np.complex128)
# Reshape to (Nx, 1, 1)
field_1a = E_x_1a[:, None, None]

total_steps_1a = int(round(Lz_1a / deltaZ_1a))
args_1a = make_args(
    Nx=Nx_1a, Ny=1, Nt=1,
    Lx=Lx_1a, Ly=1e-6, Lt=1e-12, Lz=Lz_1a,
    n2_val=0.0, beta2_val=0.0,  # linear, no dispersion
    deltaZ=deltaZ_1a, n_saves=total_steps_1a,
    pml_thickness=20,
)

print(f"Field shape: {field_1a.shape}")
print(f"Steps: {total_steps_1a}, saves: {total_steps_1a}")
print("Propagating...")
out_1a = propagate_windowed(args_1a, field_1a)
print(f"Done in {out_1a['seconds']:.1f} s")

In [ ]:
# Visualize: x-vs-z heatmap
fig, ax, _, z_1a, I_map_1a = make_transverse_vs_z_vs_I_plot(
    out_1a, args_1a, axis='x', mode='single_t', reduce='centerline',
    cmap='hot', figsize=(10, 4), title='1D Free-Space Diffraction: I(x, z)'
)
plt.show()

# Compare beam width to analytical formula
save_at_1a = np.asarray(out_1a['save_at'])
field_saved = np.asarray(out_1a['field'])
Nsave_1a = field_saved.shape[-1]

widths_sim = []
for iz in range(Nsave_1a):
    I_slice = np.abs(field_saved[:, 0, 0, iz])**2
    # Second-moment beam radius (ISO 11146): w = 2*sqrt(<x^2>_I)
    total = np.sum(I_slice)
    x2_avg = np.sum(x_1a**2 * I_slice) / total
    widths_sim.append(2 * np.sqrt(x2_avg))

widths_analytical = w0_1a * np.sqrt(1 + (save_at_1a / z_R_1a)**2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(save_at_1a * 1e3, np.array(widths_sim) * 1e6, '-', label='Simulation (2nd moment)', linewidth=2)
ax.plot(save_at_1a * 1e3, widths_analytical * 1e6, '-', label='Analytical: $w_0\\sqrt{1+(z/z_R)^2}$')
ax.set_xlabel('z (mm)'); ax.set_ylabel('Beam half-width (\u03bcm)')
ax.set_title('Gaussian Beam Diffraction: Width vs z')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 1D animation: I(x) stepping through z
make_1d_z_animation(
    np.asarray(out_1a['field']),
    coord_axis='x', coord=x_1a * 1e6,
    z=np.asarray(out_1a['save_at']) * 1e3,
    filename='sec1a_diffraction_1d.gif',
    figsize=(8, 4), fps=10,
)
print("Saved: sec1a_diffraction_1d.gif")
from IPython.display import Image, display
display(Image(filename='sec1a_diffraction_1d.gif'))

## 1b: 2+1D Free-Space Diffraction (Nx>1, Ny>1, Nt=1)

In [ ]:
# --- Tunable parameters ---
Nx_1b, Ny_1b = 128, 128
Lx_1b, Ly_1b = 400e-6, 400e-6
Lz_1b = 5e-3
w0_1b = 20e-6

X_1b, Y_1b = make_space(Lx_1b, Nx_1b, Ly_1b, Ny_1b)
x_1b, y_1b = X_1b[:, 0], Y_1b[0, :]

# 2D Gaussian beam
E_xy_1b = jnp.exp(-(X_1b**2 + Y_1b**2) / w0_1b**2).astype(jnp.complex128)
E_t_1b = cw_temp_profile_freq(jnp.array([lambda0]), jnp.array([0.0]), 1e-12, 1)
field_1b = combine_spatial_temporal(E_xy_1b, E_t_1b)

# deltaZ = z_R / 10 (reuse z_R from Section 1a)
total_steps_1b = int(round(Lz_1b / deltaZ_1a))
args_1b = make_args(
    Nx=Nx_1b, Ny=Ny_1b, Nt=1,
    Lx=Lx_1b, Ly=Ly_1b, Lt=1e-12, Lz=Lz_1b,
    n2_val=0.0, beta2_val=0.0,
    deltaZ=deltaZ_1a, n_saves=total_steps_1b,
    pml_thickness=15,
)

print(f"Field shape: {field_1b.shape}")
print(f"Steps: {total_steps_1b}")
print("Propagating...")
out_1b = propagate_windowed(args_1b, field_1b)
print(f"Done in {out_1b['seconds']:.1f} s")

In [ ]:
# 2D transverse animation over z
make_xy_z_animation(
    np.asarray(out_1b['field']),
    t_index=0,
    x=np.asarray(x_1b) * 1e6, y=np.asarray(y_1b) * 1e6,
    z=np.asarray(out_1b['save_at']) * 1e3,
    filename='sec1b_diffraction_2d.gif',
    fps=8,
)
print("Saved: sec1b_diffraction_2d.gif")
from IPython.display import Image, display
display(Image(filename='sec1b_diffraction_2d.gif'))

## 1c: 3+1D Free-Space Diffraction (Nx>1, Ny>1, Nt>1)

In [ ]:
# --- Tunable parameters ---
Nx_1c, Ny_1c, Nt_1c = 64, 64, 64
Lx_1c, Ly_1c = 400e-6, 400e-6
Lt_1c = 2e-12
Lz_1c = 5e-3
w0_1c = 20e-6
fwhm_1c = 100e-15  # 100 fs

X_1c, Y_1c = make_space(Lx_1c, Nx_1c, Ly_1c, Ny_1c)

# 2D Gaussian beam x Gaussian pulse, beta2=0
E_xy_1c = jnp.exp(-(X_1c**2 + Y_1c**2) / w0_1c**2).astype(jnp.complex128)
E_t_1c = gaussian_pulse_profile_freq(t0=0.0, fwhm=fwhm_1c, Lt=Lt_1c, Nt=Nt_1c)
field_1c = combine_spatial_temporal(E_xy_1c, E_t_1c)

# deltaZ = z_R / 10 (only diffraction active, no dispersion)
total_steps_1c = int(round(Lz_1c / deltaZ_1a))
args_1c = make_args(
    Nx=Nx_1c, Ny=Ny_1c, Nt=Nt_1c,
    Lx=Lx_1c, Ly=Ly_1c, Lt=Lt_1c, Lz=Lz_1c,
    n2_val=0.0, beta2_val=0.0,  # no nonlinearity, no dispersion
    deltaZ=deltaZ_1a, n_saves=total_steps_1c,
    pml_thickness=10,
)

print(f"Field shape: {field_1c.shape}")
print(f"Steps: {total_steps_1c}")
print("Propagating (3+1D)...")
out_1c = propagate_windowed(args_1c, field_1c)
print(f"Done in {out_1c['seconds']:.1f} s")

In [ ]:
# Animate transverse (x,y) at center time slice vs z
# Note: temporal shape unchanged since beta2=0, only spatial diffraction
make_xy_z_animation(
    np.asarray(out_1c['field']),
    t_index=Nt_1c // 2,
    x=np.linspace(-Lx_1c/2, Lx_1c/2, Nx_1c) * 1e6,
    y=np.linspace(-Ly_1c/2, Ly_1c/2, Ny_1c) * 1e6,
    z=np.asarray(out_1c['save_at']) * 1e3,
    filename='sec1c_diffraction_3d.gif',
    fps=8,
)
print("Saved: sec1c_diffraction_3d.gif")
print("Note: Temporal shape is unchanged (no dispersion). Only spatial diffraction occurs.")
from IPython.display import Image, display
display(Image(filename='sec1c_diffraction_3d.gif'))

---
# Section 2: Linear Dispersion (1+1D Temporal)

**Group velocity dispersion (GVD)**: $\beta_2$ causes frequency-dependent phase velocity, broadening pulses in time.

For a Gaussian pulse with initial width $T_0$ (half-width at $1/e$ of intensity), the width evolves as:
$$T(z) = T_0 \sqrt{1 + \left(\frac{z}{L_D}\right)^2}, \quad L_D = \frac{T_0^2}{|\beta_2|}$$

We use `Nx=1, Ny=1` to isolate the temporal dynamics.

In [ ]:
# --- Tunable parameters ---
Nt_2 = 512
Lt_2 = 4e-12         # 4 ps window
fwhm_2 = 100e-15     # 100 fs pulse
beta2_2 = 22e-27      # normal GVD

# T0 = FWHM / (2*sqrt(2*ln2)) for a Gaussian
T0_2 = fwhm_2 / (2 * np.sqrt(2 * np.log(2)))
L_D_2 = T0_2**2 / abs(beta2_2)
Lz_2 = 4 * L_D_2    # propagate through several dispersion lengths
deltaZ_2 = L_D_2 / 1000  # step size ~ 1/10 of dispersion length

print(f"T0 = {T0_2*1e15:.1f} fs")
print(f"Dispersion length L_D = {L_D_2*1e3:.3f} mm")
print(f"deltaZ = L_D/10 = {deltaZ_2*1e3:.3f} mm")
print(f"Propagation distance Lz = {Lz_2*1e3:.3f} mm = {Lz_2/L_D_2:.1f} L_D")
print(f"Total steps ~ {int(round(Lz_2 / deltaZ_2))}")

In [ ]:
# Gaussian pulse, no nonlinearity
E_t_2 = gaussian_pulse_profile_freq(t0=0.0, fwhm=fwhm_2, Lt=Lt_2, Nt=Nt_2)
field_2 = E_t_2[None, None, :]  # (1, 1, Nt)

total_steps_2 = int(round(Lz_2 / deltaZ_2))
args_2 = make_args(
    Nx=1, Ny=1, Nt=Nt_2,
    Lx=1e-6, Ly=1e-6, Lt=Lt_2, Lz=Lz_2,
    n2_val=0.0, beta2_val=beta2_2,
    deltaZ=deltaZ_2, n_saves=total_steps_2,
)

print(f"Field shape: {field_2.shape}")
print(f"Steps = saves: {total_steps_2}")
print("Propagating...")
out_2 = propagate_windowed(args_2, field_2)
print(f"Done in {out_2['seconds']:.1f} s")

In [ ]:
# t-vs-z heatmap
fig, ax, t_2, z_2, I_map_2 = make_temporal_vs_z_plot(
    out_2, args_2, reduce='centerline',
    cmap='hot', figsize=(10, 4),
    title='Dispersive Broadening: I(t, z)'
)
plt.show()

# 1D animation: I(t) vs z
dt_2 = Lt_2 / Nt_2
t_axis_2 = np.linspace(-Lt_2/2, Lt_2/2, Nt_2)
make_1d_z_animation(
    np.asarray(out_2['field']),
    coord_axis='t', coord=t_axis_2 * 1e15,  # in fs
    z=np.asarray(out_2['save_at']) * 1e3,
    filename='sec2_dispersion.gif',
    figsize=(8, 4), fps=10,
)
print("Saved: sec2_dispersion.gif")
from IPython.display import Image, display
display(Image(filename='sec2_dispersion.gif'))

In [ ]:
# Compare pulse width to analytical broadening
save_at_2 = np.asarray(out_2['save_at'])
field_saved_2 = np.asarray(out_2['field'])

widths_2_sim = []
for iz in range(field_saved_2.shape[-1]):
    I_t = np.abs(field_saved_2[0, 0, :, iz])**2
    I_t_norm = I_t / I_t.max()
    above = I_t_norm >= 0.5  # FWHM
    if above.any():
        widths_2_sim.append((t_axis_2[above][-1] - t_axis_2[above][0]))
    else:
        widths_2_sim.append(np.nan)

# Analytical FWHM broadening
fwhm_analytical_2 = fwhm_2 * np.sqrt(1 + (save_at_2 / L_D_2)**2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(save_at_2 * 1e3, np.array(widths_2_sim) * 1e15, 'o', label='Simulation (FWHM)', markersize=4)
ax.plot(save_at_2 * 1e3, fwhm_analytical_2 * 1e15, '-', label='Analytical: FWHM$\\cdot\\sqrt{1+(z/L_D)^2}$')
ax.set_xlabel('z (mm)'); ax.set_ylabel('Pulse FWHM (fs)')
ax.set_title('Dispersive Pulse Broadening')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
# Section 3: Nonlinear, No Dispersion (1+1D)

**Self-phase modulation (SPM)**: The Kerr effect adds an intensity-dependent phase $\phi_{NL}(t) = \gamma P(t) z$ without changing the temporal intensity profile. This broadens the spectrum.

$$L_{NL} = \frac{1}{\gamma P_0}$$

With $\beta_2 = 0$ and pure Kerr ($f_R = 0$, `sw=0`), the temporal shape $|A(t)|^2$ is invariant.

In [ ]:
# --- Tunable parameters ---
Nt_3 = 512
Lt_3 = 4e-12
fwhm_3 = 100e-15
P0_3 = 1e3           # 1 kW peak power

# For 1D temporal (Nx=Ny=1), the solver uses bulk gamma, so the field
# must be in sqrt(intensity). Convert power -> intensity via A_eff.
I0_3 = P0_3 / A_eff  # peak intensity [W/m^2]
L_NL_3 = 1.0 / (gamma * I0_3)
Lz_3 = 4 * L_NL_3
deltaZ_3 = L_NL_3 / 10

print(f"gamma (bulk) = {gamma:.4e} m/W")
print(f"P0 = {P0_3:.0f} W => I0 = P0/A_eff = {I0_3:.3e} W/m^2")
print(f"Nonlinear length L_NL = {L_NL_3:.3f} m")
print(f"deltaZ = L_NL/10 = {deltaZ_3:.4f} m")
print(f"Propagation distance Lz = {Lz_3:.3f} m = {Lz_3/L_NL_3:.1f} L_NL")
print(f"Total steps ~ {int(round(Lz_3 / deltaZ_3))}")

In [ ]:
# Gaussian pulse with peak intensity I0 (NOT power — solver uses bulk gamma)
E_t_3 = gaussian_pulse_profile_freq(t0=0.0, fwhm=fwhm_3, Lt=Lt_3, Nt=Nt_3)
E_t_3 = E_t_3 * jnp.sqrt(I0_3)  # scale to peak intensity
field_3 = E_t_3[None, None, :]

total_steps_3 = int(round(Lz_3 / deltaZ_3))
n_saves_3 = total_steps_3
args_3 = make_args(
    Nx=1, Ny=1, Nt=Nt_3,
    Lx=1e-6, Ly=1e-6, Lt=Lt_3, Lz=Lz_3,
    n2_val=n2, beta2_val=0.0,  # nonlinear, no dispersion
    fr=0.0, sw=0,              # pure Kerr
    deltaZ=deltaZ_3, n_saves=n_saves_3,
)

print(f"Steps: {int(round(Lz_3 / deltaZ_3))}, saves: {n_saves_3}")
print("Propagating (SPM only)...")
out_3 = propagate_windowed(args_3, field_3)
print(f"Done in {out_3['seconds']:.1f} s")

In [ ]:
# I(t) vs z: should be constant shape
fig, ax, _, _, _ = make_temporal_vs_z_plot(
    out_3, args_3, reduce='centerline',
    cmap='hot', figsize=(10, 4),
    title='SPM: I(t, z) \u2014 Temporal shape preserved'
)
plt.show()

# Spectral evolution: |A~(omega)|^2 vs z
field_saved_3 = np.asarray(out_3['field'])
save_at_3 = np.asarray(out_3['save_at'])
dt_3 = Lt_3 / Nt_3
omega_3 = 2 * np.pi * np.fft.fftfreq(Nt_3, dt_3)
omega_3_shifted = np.fft.fftshift(omega_3)

# Build spectral heatmap
spec_map = np.zeros((Nt_3, field_saved_3.shape[-1]))
for iz in range(field_saved_3.shape[-1]):
    E_w = np.fft.fftshift(np.fft.fft(field_saved_3[0, 0, :, iz]))
    spec_map[:, iz] = np.abs(E_w)**2

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
extent = [save_at_3.min()*1e3, save_at_3.max()*1e3,
          omega_3_shifted.min()*1e-12, omega_3_shifted.max()*1e-12]
ax.imshow(spec_map, origin='lower', aspect='auto', extent=extent, cmap='hot')
ax.set_xlabel('z (mm)'); ax.set_ylabel('\u03c9 (THz)')
ax.set_title('SPM: Spectral broadening |\u00c3(\u03c9, z)|\u00b2')
plt.show()

---
# Section 4: Nonlinear + Dispersion (1+1D)

When SPM and GVD compete, the outcome depends on the dispersion sign:
- **Normal dispersion** ($\beta_2 > 0$): SPM + GVD cause pulse distortion and breakup
- **Anomalous dispersion** ($\beta_2 < 0$): balance leads to **soliton** formation

The **soliton number** $N^2 = \gamma P_0 T_0^2 / |\beta_2|$ governs the dynamics. $N=1$ gives a fundamental soliton:
$$A(0, t) = \sqrt{P_0}\,\mathrm{sech}(t/T_0), \quad P_0 = \frac{|\beta_2|}{\gamma T_0^2}$$

## 4a: Gaussian Pulse, Normal Dispersion

In [ ]:
# --- Tunable parameters ---
Nt_4a = 512
Lt_4a = 8e-12
fwhm_4a = 100e-15
P0_4a = 1e3
beta2_4a = 22e-27   # normal (positive)

T0_4a = fwhm_4a / (2 * np.sqrt(2 * np.log(2)))
L_D_4a = T0_4a**2 / abs(beta2_4a)

# Convert power to intensity for 1D
I0_4a = P0_4a / A_eff
L_NL_4a = 1.0 / (gamma * I0_4a)
N_4a = np.sqrt(L_D_4a / L_NL_4a)
Lz_4a = 3 * L_D_4a

# Step size from shortest characteristic length
L_min_4a = min(L_D_4a, L_NL_4a)
deltaZ_4a = L_min_4a / 10

print(f"L_D = {L_D_4a*1e3:.3f} mm, L_NL = {L_NL_4a:.3f} m")
print(f"Soliton number N = {N_4a:.2f}")
print(f"deltaZ = min(L_D,L_NL)/10 = {deltaZ_4a*1e3:.3f} mm")
print(f"Total steps ~ {int(round(Lz_4a / deltaZ_4a))}")

E_t_4a = gaussian_pulse_profile_freq(t0=0.0, fwhm=fwhm_4a, Lt=Lt_4a, Nt=Nt_4a)
E_t_4a = E_t_4a * jnp.sqrt(I0_4a)  # intensity, not power
field_4a = E_t_4a[None, None, :]

total_steps_4a = int(round(Lz_4a / deltaZ_4a))
n_saves_4a = total_steps_4a
args_4a = make_args(
    Nx=1, Ny=1, Nt=Nt_4a,
    Lx=1e-6, Ly=1e-6, Lt=Lt_4a, Lz=Lz_4a,
    n2_val=n2, beta2_val=beta2_4a,
    fr=0.0, sw=0,
    deltaZ=deltaZ_4a, n_saves=n_saves_4a,
)

print("Propagating (normal dispersion + Kerr)...")
out_4a = propagate_windowed(args_4a, field_4a)
print(f"Done in {out_4a['seconds']:.1f} s")

In [ ]:
fig, ax, _, _, _ = make_temporal_vs_z_plot(
    out_4a, args_4a, reduce='centerline',
    cmap='hot', figsize=(10, 4),
    title='Normal Dispersion + Kerr: I(t, z) \u2014 Pulse distortion'
)
plt.show()

## 4b: Fundamental Soliton (Anomalous Dispersion)

In [ ]:
# --- Tunable parameters ---
Nt_4b = 512
Lt_4b = 8e-12
T0_4b = 100e-15      # sech width parameter
beta2_4b = -22e-27    # anomalous (negative)

# Soliton condition: P0 = |beta2| / (gamma * T0^2)
# Note: this gives peak INTENSITY [W/m^2], which is correct for the bulk-gamma solver.
P0_4b = abs(beta2_4b) / (gamma * T0_4b**2)
L_D_4b = T0_4b**2 / abs(beta2_4b)
# For N=1 soliton, L_NL = L_D
Lz_4b = 5 * L_D_4b
deltaZ_4b = L_D_4b / 1000

print(f"Soliton peak intensity I0 = {P0_4b:.3e} W/m^2")
print(f"  (equivalent power P = I0*A_eff = {P0_4b*A_eff:.1f} W)")
print(f"L_D = L_NL = {L_D_4b*1e3:.3f} mm")
print(f"deltaZ = L_D/10 = {deltaZ_4b*1e3:.3f} mm")
print(f"Propagation: {Lz_4b/L_D_4b:.1f} L_D, steps ~ {int(round(Lz_4b/deltaZ_4b))}")

# sech(t/T0) initial condition
dt_4b = Lt_4b / Nt_4b
t_4b = np.linspace(-Lt_4b/2, Lt_4b/2, Nt_4b)
E_t_4b = np.sqrt(P0_4b) / np.cosh(t_4b / T0_4b)
field_4b = jnp.asarray(E_t_4b[None, None, :], dtype=jnp.complex128)

total_steps_4b = int(round(Lz_4b / deltaZ_4b))
n_saves_4b = total_steps_4b
args_4b = make_args(
    Nx=1, Ny=1, Nt=Nt_4b,
    Lx=1e-6, Ly=1e-6, Lt=Lt_4b, Lz=Lz_4b,
    n2_val=n2, beta2_val=beta2_4b,
    fr=0.0, sw=0,
    deltaZ=deltaZ_4b, n_saves=n_saves_4b,
)

print("Propagating (fundamental soliton)...")
out_4b = propagate_windowed(args_4b, field_4b)
print(f"Done in {out_4b['seconds']:.1f} s")

In [ ]:
# t-vs-z heatmap: should show straight horizontal bands (shape-preserving)
fig, ax, _, _, _ = make_temporal_vs_z_plot(
    out_4b, args_4b, reduce='centerline',
    cmap='hot', figsize=(10, 4),
    title='Fundamental Soliton: I(t, z) \u2014 Shape preserved'
)
plt.show()

# Overlay input vs output pulse shape
field_saved_4b = np.asarray(out_4b['field'])
I_in_4b = np.abs(field_saved_4b[0, 0, :, 0])**2
I_out_4b = np.abs(field_saved_4b[0, 0, :, -1])**2

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_4b * 1e15, I_in_4b / I_in_4b.max(), '-', label='Input')
ax.plot(t_4b * 1e15, I_out_4b / I_out_4b.max(), '--', label='Output')
ax.plot(t_4b * 1e15, 1.0 / np.cosh(t_4b / T0_4b)**2, ':', label='sech\u00b2 (analytical)', alpha=0.7)
ax.set_xlabel('t (fs)'); ax.set_ylabel('Normalized intensity')
ax.set_title('Soliton: Input vs Output vs Analytical')
ax.set_xlim([-500, 500])
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
# Section 5: Linear Fiber Propagation (2+1D)

**Waveguiding**: Total internal reflection confines light in a fiber core where $n_{\text{core}} > n_{\text{clad}}$.

- **GRIN fiber**: Parabolic refractive index profile $n(r) = n_{\text{core}}\sqrt{1 - 2\Delta(r/a)^\alpha}$ with $\alpha=2$. Modes resemble Hermite-Gaussians.
- **Step-index fiber**: Flat core + cladding. Modes are Bessel-like.

Each mode has a propagation constant $\beta_m$ and a transverse profile. The eigenmode solver (`solve_modes`) computes these numerically.

In [ ]:
# --- Tunable parameters ---
Nx_5, Ny_5 = 128, 128
Lx_5, Ly_5 = 150e-6, 150e-6
Lz_5 = 5e-2          # 5 cm
r_core_5 = 25e-6     # 25 um core radius
n_modes_5 = 10
lambda_nm = int(lambda0 * 1e9)
r_core_um = int(r_core_5 * 1e6)

dx_5, dy_5 = Lx_5 / Nx_5, Ly_5 / Ny_5
X_5, Y_5 = make_space(Lx_5, Nx_5, Ly_5, Ny_5)
x_5, y_5 = X_5[:, 0], Y_5[0, :]

# Characteristic length for guided modes: self-imaging period of GRIN fiber
# P_si = pi * r_core / sqrt(2*Delta), Delta = (n_core - n_clad) / n_core
Delta_5 = (n_core - n_clad) / n_core
L_beat_5 = np.pi * r_core_5 / np.sqrt(2 * Delta_5)
deltaZ_5 = L_beat_5 / 10

print(f"Delta = {Delta_5:.5f}")
print(f"GRIN self-imaging period L_beat = {L_beat_5*1e3:.3f} mm")
print(f"deltaZ = L_beat/10 = {deltaZ_5*1e3:.3f} mm")
print(f"Steps for Lz={Lz_5*1e2:.0f} cm: ~ {int(round(Lz_5 / deltaZ_5))}")

In [ ]:
# Build refractive index profiles
n_grin_5 = make_polynomial_n(X_5, Y_5, n_core=n_core, n_clad=n_clad, r_core=r_core_5, alpha=2)
n_step_5 = make_supergauss_index(X_5, Y_5, n_core=n_core, n_clad=n_clad, r_core=r_core_5, m=20)

# Visualize both profiles
ext_5 = [float(x_5.min())*1e6, float(x_5.max())*1e6,
         float(y_5.min())*1e6, float(y_5.max())*1e6]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im0 = axes[0].imshow(np.asarray(n_grin_5).T, origin='lower', extent=ext_5, cmap='viridis')
axes[0].set_title('GRIN Profile (\u03b1=2)'); axes[0].set_xlabel('x (\u03bcm)'); axes[0].set_ylabel('y (\u03bcm)')
plt.colorbar(im0, ax=axes[0], label='n')

im1 = axes[1].imshow(np.asarray(n_step_5).T, origin='lower', extent=ext_5, cmap='viridis')
axes[1].set_title('Step-Index Profile (m=20)'); axes[1].set_xlabel('x (\u03bcm)'); axes[1].set_ylabel('y (\u03bcm)')
plt.colorbar(im1, ax=axes[1], label='n')

plt.tight_layout(); plt.show()

In [ ]:
# Solve modes (cached with overwrite=False)
folder_grin = Path(f"modes_grin_Nx{Nx_5}_r{r_core_um}um_{lambda_nm}nm")
folder_step = Path(f"modes_step_Nx{Nx_5}_r{r_core_um}um_{lambda_nm}nm")

print("Solving GRIN modes...")
solve_modes(n_xy=n_grin_5, n_ref=n_core, x=x_5, y=y_5,
            lambda0=lambda0, n_modes=n_modes_5, folder=folder_grin)

print("Solving step-index modes...")
solve_modes(n_xy=n_step_5, n_ref=n_core, x=x_5, y=y_5,
            lambda0=lambda0, n_modes=n_modes_5, folder=folder_step)

In [ ]:
# Display modes
print("GRIN fiber modes:")
fig_grin = plot_modes_gallery(folder_grin, ncols=5, suptitle='GRIN Fiber Modes')
plt.show()

print("Step-index fiber modes:")
fig_step = plot_modes_gallery(folder_step, ncols=5, suptitle='Step-Index Fiber Modes')
plt.show()

In [ ]:
# Three excitation cases for GRIN fiber (linear, CW)
n_xyomega_grin = jnp.tile(n_grin_5[:, :, None], (1, 1, 1))
E_t_cw = cw_temp_profile_freq(jnp.array([lambda0]), jnp.array([0.0]), 1e-12, 1)

power_5 = 1.0  # 1 W (linear, doesn't matter for shape)

# Case A: Single mode (fundamental)
E_xy_A, idx_A, c_A = make_source_from_files(folder_grin, heading='mode', weights={0: 1.0})
E_xy_A, _ = norm_scale_field_weights(E_xy_A, idx_A, c_A, power=power_5, dx=dx_5, dy=dy_5)
field_5A = combine_spatial_temporal(E_xy_A, E_t_cw)

# Case B: Few mode (0, 1, 2)
E_xy_B, idx_B, c_B = make_source_from_files(folder_grin, heading='mode',
                                              weights={0: 1.0, 1: 0.7, 2: 0.5})
E_xy_B, _ = norm_scale_field_weights(E_xy_B, idx_B, c_B, power=power_5, dx=dx_5, dy=dy_5)
field_5B = combine_spatial_temporal(E_xy_B, E_t_cw)

# Case C: Speckle (all modes, random weights and phases)
rng = np.random.default_rng(42)
n_avail = min(n_modes_5, len(list(folder_grin.glob('mode_*.npz'))))
weights_C = {i: float(rng.uniform(0.5, 1.5)) * np.exp(1j * rng.uniform(0, 2*np.pi))
             for i in range(n_avail)}
E_xy_C, idx_C, c_C = make_source_from_files(folder_grin, heading='mode', weights=weights_C)
E_xy_C, _ = norm_scale_field_weights(E_xy_C, idx_C, c_C, power=power_5, dx=dx_5, dy=dy_5)
field_5C = combine_spatial_temporal(E_xy_C, E_t_cw)

print(f"Single mode field: {field_5A.shape}")
print(f"Few mode field:    {field_5B.shape}")
print(f"Speckle field:     {field_5C.shape}")

In [ ]:
# Propagate all three cases
total_steps_5 = int(round(Lz_5 / deltaZ_5))
n_saves_5 = total_steps_5
args_5 = make_args(
    Nx=Nx_5, Ny=Ny_5, Nt=1,
    Lx=Lx_5, Ly=Ly_5, Lt=1e-12, Lz=Lz_5,
    n_xyomega=n_grin_5,
    n2_val=0.0, beta2_val=0.0,  # linear
    deltaZ=deltaZ_5, n_saves=n_saves_5,
    pml_thickness=15,
)

print(f"deltaZ = {deltaZ_5*1e3:.3f} mm, saves = {n_saves_5}")
print("Case A (single mode)...")
out_5A = propagate_windowed(args_5, field_5A)
print(f"  Done in {out_5A['seconds']:.1f} s")

print("Case B (few modes)...")
out_5B = propagate_windowed(args_5, field_5B)
print(f"  Done in {out_5B['seconds']:.1f} s")

print("Case C (speckle)...")
out_5C = propagate_windowed(args_5, field_5C)
print(f"  Done in {out_5C['seconds']:.1f} s")

In [ ]:
# Animate each case
z_5 = np.asarray(out_5A['save_at']) * 1e3
x5_um = np.asarray(x_5) * 1e6
y5_um = np.asarray(y_5) * 1e6

for label, out, fname in [
    ('A_single_mode', out_5A, 'sec5_single_mode.gif'),
    ('B_few_modes', out_5B, 'sec5_few_modes.gif'),
    ('C_speckle', out_5C, 'sec5_speckle.gif'),
]:
    make_xy_z_animation(
        np.asarray(out['field']), t_index=0,
        x=x5_um, y=y5_um, z=z_5,
        filename=fname, fps=8,
    )
    print(f"Saved: {fname}")

from IPython.display import Image, display
print("\nSingle mode (should propagate unchanged):")
display(Image(filename='sec5_single_mode.gif'))
print("\nFew modes (beating pattern):")
display(Image(filename='sec5_few_modes.gif'))
print("\nSpeckle (complex interference):")
display(Image(filename='sec5_speckle.gif'))

---
# Section 6: Nonlinear Fiber Propagation (2+1D)

### Key Nonlinear Length Scales

- **Critical power**: $P_{cr} = \lambda_0^2 / (2\pi n_2)$ — self-focusing threshold
- **Nonlinear length**: $L_{NL} = 1/(\gamma P_0)$
- **Dispersion length**: $L_D = T_0^2 / |\beta_2|$

When the input power approaches $P_{cr}$, transverse spatial dynamics become strongly nonlinear.

In [ ]:
# Compute nonlinear length scales
P_cr = lambda0**2 / (2 * np.pi * n2)
print(f"Critical power P_cr = {P_cr/1e6:.2f} MW")

# Use power = P_cr / 2 (below self-focusing threshold)
power_6 = P_cr / 2
L_NL_6 = 1.0 / (gamma * power_6)
print(f"\nAt P = P_cr/2 = {power_6/1e6:.3f} MW:")
print(f"  L_NL = {L_NL_6*1e3:.4f} mm")

In [ ]:
# Nonlinear propagation with same GRIN fiber modes from Section 5
# Reuse modes: single mode excitation at P_cr/2
E_xy_6, idx_6, c_6 = make_source_from_files(folder_grin, heading='mode', weights={0: 1.0})
E_xy_6, _ = norm_scale_field_weights(E_xy_6, idx_6, c_6, power=power_6, dx=dx_5, dy=dy_5)
field_6 = combine_spatial_temporal(E_xy_6, E_t_cw)

# L_beat is still the shortest characteristic length (L_NL >> L_beat for fiber modes)
total_steps_6 = int(round(Lz_5 / deltaZ_5))
n_saves_6 = total_steps_6
args_6 = make_args(
    Nx=Nx_5, Ny=Ny_5, Nt=1,
    Lx=Lx_5, Ly=Ly_5, Lt=1e-12, Lz=Lz_5,
    n_xyomega=n_grin_5,
    n2_val=n2, beta2_val=0.0,  # nonlinear, CW
    fr=0.0, sw=0,
    deltaZ=deltaZ_5, n_saves=n_saves_6,
    pml_thickness=15,
)

print(f"deltaZ = {deltaZ_5*1e3:.3f} mm (from L_beat)")
print("Nonlinear propagation (CW, P = P_cr/2)...")
out_6 = propagate_windowed(args_6, field_6)
print(f"Done in {out_6['seconds']:.1f} s")

In [ ]:
# Animate nonlinear vs linear
make_xy_z_animation(
    np.asarray(out_6['field']), t_index=0,
    x=x5_um, y=y5_um, z=np.asarray(out_6['save_at']) * 1e3,
    filename='sec6_nonlinear.gif', fps=8,
)
print("Saved: sec6_nonlinear.gif")

# Compare peak intensity vs z (linear vs nonlinear)
field_lin = np.asarray(out_5A['field'])
field_nl = np.asarray(out_6['field'])
save_at_5 = np.asarray(out_5A['save_at'])

# Normalize linear to same input power for fair comparison
scale_factor = np.sqrt(power_6 / power_5)

peak_lin = np.array([np.max(np.abs(field_lin[:, :, 0, iz])**2) for iz in range(field_lin.shape[-1])])
peak_nl = np.array([np.max(np.abs(field_nl[:, :, 0, iz])**2) for iz in range(field_nl.shape[-1])])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(save_at_5 * 1e2, peak_lin * scale_factor**2, '-', label='Linear (scaled)')
ax.plot(save_at_5 * 1e2, peak_nl, '--', label='Nonlinear (P=P_cr/2)')
ax.set_xlabel('z (cm)'); ax.set_ylabel('Peak |E|\u00b2')
ax.set_title('Peak Intensity: Linear vs Nonlinear')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

from IPython.display import Image, display
display(Image(filename='sec6_nonlinear.gif'))

---
# Section 7: Full 3+1D Nonlinear Pulse Propagation

Combining all effects: spatial waveguiding, temporal dispersion, Kerr nonlinearity. This is the **full GNLSE** with 3 spatial + 1 temporal dimensions.

We use a GRIN fiber, fundamental mode excitation with a Gaussian pulse.

In [ ]:
# --- Tunable parameters ---
Nx_7, Ny_7, Nt_7 = 64, 64, 64
Lx_7, Ly_7 = 150e-6, 150e-6
Lt_7 = 2e-12
Lz_7 = 1e-2          # 1 cm
fwhm_7 = 100e-15
power_7 = 0.5e6      # 0.5 MW

dx_7, dy_7, dt_7 = Lx_7/Nx_7, Ly_7/Ny_7, Lt_7/Nt_7
X_7, Y_7 = make_space(Lx_7, Nx_7, Ly_7, Ny_7)
x_7, y_7 = X_7[:, 0], Y_7[0, :]

# GRIN fiber index profile (on this grid)
n_grin_7 = make_polynomial_n(X_7, Y_7, n_core=n_core, n_clad=n_clad, r_core=r_core_5, alpha=2)

# Solve modes on this grid
folder_grin_7 = Path(f"modes_grin_Nx{Nx_7}_r{r_core_um}um_{lambda_nm}nm")
solve_modes(n_xy=n_grin_7, n_ref=n_core, x=x_7, y=y_7,
            lambda0=lambda0, n_modes=4, folder=folder_grin_7)

# Source: fundamental mode + Gaussian pulse
E_xy_7, idx_7, c_7 = make_source_from_files(folder_grin_7, heading='mode', weights={0: 1.0})
E_xy_7, _ = norm_scale_field_weights(E_xy_7, idx_7, c_7, power=1.0, dx=dx_7, dy=dy_7)
E_t_7 = gaussian_pulse_profile_freq(t0=0.0, fwhm=fwhm_7, Lt=Lt_7, Nt=Nt_7)
E_t_7 = E_t_7 * jnp.sqrt(power_7)
field_7 = combine_spatial_temporal(E_xy_7, E_t_7)

# deltaZ from beat length (shortest characteristic length in GRIN fiber)
# L_beat_5 computed in Section 5, same fiber parameters
deltaZ_7 = L_beat_5 / 10
total_steps_7 = int(round(Lz_7 / deltaZ_7))
n_saves_7 = total_steps_7
args_7 = make_args(
    Nx=Nx_7, Ny=Ny_7, Nt=Nt_7,
    Lx=Lx_7, Ly=Ly_7, Lt=Lt_7, Lz=Lz_7,
    n_xyomega=n_grin_7,
    n2_val=n2, beta2_val=beta2,
    fr=0.0, sw=0,  # pure Kerr for speed
    deltaZ=deltaZ_7, n_saves=n_saves_7,
    pml_thickness=10,
    precision="fp32",
)

print(f"Field shape: {field_7.shape}")
print(f"Field size: {field_7.nbytes/1e6:.1f} MB")
print(f"deltaZ = {deltaZ_7*1e3:.3f} mm (from L_beat), steps ~ {int(round(Lz_7/deltaZ_7))}")
print("Propagating (full 3+1D)...")
out_7 = propagate_windowed(args_7, field_7)
print(f"Done in {out_7['seconds']:.1f} s")

In [ ]:
field_7_saved = np.asarray(out_7['field'])

# Input (z=0): x-y at peak time
make_xy_t_animation(
    field_7_saved, z_index=0,
    x=np.asarray(x_7)*1e6, y=np.asarray(y_7)*1e6,
    t=np.linspace(-Lt_7/2, Lt_7/2, Nt_7)*1e15,
    filename='sec7_input_xy_t.gif', fps=10, norm='per_frame',
)
print("Saved: sec7_input_xy_t.gif")

# Output (z=-1): x-y at peak time
make_xy_t_animation(
    field_7_saved, z_index=-1,
    x=np.asarray(x_7)*1e6, y=np.asarray(y_7)*1e6,
    t=np.linspace(-Lt_7/2, Lt_7/2, Nt_7)*1e15,
    filename='sec7_output_xy_t.gif', fps=10, norm='per_frame',
)
print("Saved: sec7_output_xy_t.gif")

# Compare temporal pulse shape: input vs output (at spatial center)
t_7 = np.linspace(-Lt_7/2, Lt_7/2, Nt_7)
I_in_7 = np.abs(field_7_saved[Nx_7//2, Ny_7//2, :, 0])**2
I_out_7 = np.abs(field_7_saved[Nx_7//2, Ny_7//2, :, -1])**2

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_7 * 1e15, I_in_7 / I_in_7.max(), '-', label='Input (z=0)')
ax.plot(t_7 * 1e15, I_out_7 / I_out_7.max(), '--', label='Output (z=Lz)')
ax.set_xlabel('t (fs)'); ax.set_ylabel('Normalized |E|\u00b2')
ax.set_title('3+1D: Temporal Pulse Shape at Fiber Center')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

from IPython.display import Image, display
print("Input (x,y) vs time:")
display(Image(filename='sec7_input_xy_t.gif'))
print("Output (x,y) vs time:")
display(Image(filename='sec7_output_xy_t.gif'))

---
# Section 8: Soliton & Multisoliton Optimization

## Goal

Given a pulse at the correct soliton power, **optimize its temporal shape** so that it propagates with minimal distortion. The optimizer should converge toward the analytical soliton profile $\text{sech}(t/T_0)$.

## Soliton physics

- **N=1 (fundamental)**: $A(0,t) = \sqrt{P_0}\,\text{sech}(t/T_0)$ with $P_0 = |\beta_2|/(\gamma T_0^2)$. Shape-preserving for all $z$.
- **N=2 (second-order)**: $P_0 = 4|\beta_2|/(\gamma T_0^2)$. Periodic breathing with soliton period $z_{\text{sp}} = \frac{\pi}{2} L_D$; returns to initial shape at $z = z_{\text{sp}}$.

## Optimization approach

- **Windowed propagation** for memory-efficient gradients (`windowed_grad`).
- **Dimensionless loss**: relative L2 error between normalized output and input intensities:
  $$L = \frac{\sum_i |I_{\text{out,norm}}(t_i) - I_{\text{in,norm}}(t_i)|^2}{\sum_i |I_{\text{in,norm}}(t_i)|^2}$$
- **Moving target** (self-consistency): each step recomputes $I_{\text{in,norm}}$ from the current field. JAX treats this as a stop-gradient constant, so gradients flow only through propagation. Converges when output $\approx$ input (soliton fixed point).
- **Two initializations**: (a) Gaussian pulse, (b) random temporal Hermite-Gaussian basis.
- **Two soliton orders**: N=1 ($L_z = 5 L_D$) and N=2 ($L_z = z_{\text{sp}}$).
- **SGD** with adaptive learning rate: $\eta = \alpha\,|L|/\|\nabla L\|^2$, plus power renormalization.

In [ ]:
# Free memory from prior sections before heavy optimization
import gc
for _name in ['out_7', 'field_7_saved', 'out_5A', 'out_5B', 'out_5C',
              'out_6A', 'out_6B', 'out_6C']:
    if _name in dir():
        exec(f'del {_name}')
gc.collect()
jax.clear_caches()
print('Cleared prior section outputs and JIT caches.')

# --- Section 8: Parameters and temporal HG basis ---
Nt_8 = 1024
Lt_8 = 8e-12         # 8 ps window
T0_8 = 100e-15       # sech width parameter
beta2_8 = -22e-27    # anomalous

dt_8 = Lt_8 / Nt_8
t_8 = np.linspace(-Lt_8/2, Lt_8/2, Nt_8)

# Soliton parameters
L_D_8 = T0_8**2 / abs(beta2_8)
P0_N1 = abs(beta2_8) / (gamma * T0_8**2)          # N=1 peak intensity
P0_N2 = 4 * abs(beta2_8) / (gamma * T0_8**2)      # N=2 peak intensity
z_sp = (np.pi / 2) * L_D_8                         # soliton period
Lz_N1 = 5 * L_D_8                                  # enough to distinguish soliton from non-soliton
Lz_N2 = z_sp                                       # one soliton period (N=2 returns to initial shape)
deltaZ_8 = L_D_8 / 100

n_windows_8 = 10
n_gd_steps_8 = 1000

# Temporal Hermite-Gaussian basis builder
from numpy.polynomial.hermite_e import hermeval

def make_temporal_hg_basis(t, w0, n_modes):
    """Build L2-normalized probabilist's Hermite-Gaussian basis on grid t."""
    xi = t / w0
    gauss = np.exp(-xi**2 / 2)
    basis = np.zeros((n_modes, len(t)))
    for n in range(n_modes):
        coeffs = np.zeros(n + 1)
        coeffs[n] = 1.0
        psi = hermeval(xi, coeffs) * gauss
        norm = np.sqrt(np.sum(psi**2) * (t[1] - t[0]))
        basis[n] = psi / norm
    return basis

w0_hg = 2 * T0_8
n_hg_modes = 8
hg_basis = make_temporal_hg_basis(t_8, w0_hg, n_hg_modes)

print(f"N=1: P0 = {P0_N1:.3e} W/m^2, Lz = {Lz_N1*1e3:.2f} mm ({Lz_N1/L_D_8:.0f} L_D)")
print(f"N=2: P0 = {P0_N2:.3e} W/m^2, Lz = {Lz_N2*1e3:.2f} mm (z_sp = pi/2 * L_D)")
print(f"L_D = {L_D_8*1e3:.3f} mm, deltaZ = {deltaZ_8*1e6:.1f} um")
print(f"HG basis: {n_hg_modes} modes, w0 = {w0_hg*1e15:.0f} fs")
print(f"GD steps: {n_gd_steps_8}, windows: {n_windows_8}")

In [ ]:
# --- Loss function and reusable optimization loop ---

def make_soliton_loss_fn(replicate, dt):
    """Unnormalized L2 between output and target intensities.
    
    Physical units (W/m^2)^2 * s keep gradient well-scaled relative
    to the field, matching the proven pattern from windowed_optimization_example.
    """
    def loss_fn(field_kwo_final, I_target):
        field_xyt = _materialize_xyt(field_kwo_final, replicate)
        I_out = jnp.abs(field_xyt[0, 0, :])**2
        return jnp.sum((I_out - I_target)**2) * dt
    return loss_fn


def run_soliton_optimization(field_init, args, ctx, replicate, n_steps,
                             n_windows, lr_alpha=0.01, print_every=20):
    """Self-consistency gradient descent for soliton shape.

    Moving target: each step, the target is the CURRENT field's intensity
    (recomputed but NOT differentiated through). At the fixed point,
    I_out == I_in -- the field propagates unchanged (= soliton).

    Unnormalized loss in physical units for well-scaled gradients,
    with adaptive LR + safety cap matching windowed_optimization_example.
    Returns (A_opt, history)."""
    dt_val = args['Lt'] / args['Nt']
    CD = jnp.complex128

    base_loss = make_soliton_loss_fn(replicate, dt_val)

    A_opt = np.array(field_init, copy=True)
    P0_fixed = float(jnp.sum(jnp.abs(A_opt)**2) * dt_val)

    history = {'step': [], 'loss': [], 'eps': [], 'lr': []}

    for step in range(n_steps):
        # Moving target: current field's intensity (NOT differentiated through)
        I_in = jnp.abs(jnp.asarray(A_opt[0, 0, :]))**2
        I_in_sq_dt = float(jnp.sum(I_in**2) * dt_val)

        def loss_fn(field_kwo_final, _I_target=I_in):
            return base_loss(field_kwo_final, _I_target)

        wg = windowed_grad(loss_fn, args, A_opt,
                           n_windows=n_windows, ctx=ctx)
        loss_val = float(wg['loss'])
        eps = loss_val / (I_in_sq_dt + 1e-30)  # dimensionless self-consistency

        # Adaptive LR (same formula as windowed_optimization_example)
        gnorm2 = float(jnp.sum(jnp.abs(wg['grad'])**2))
        lr = lr_alpha * abs(loss_val) / (gnorm2 + 1e-30)

        # Safety cap: limit relative step to 1% of field norm
        A_kwo = jnp.fft.fftn(jnp.asarray(A_opt, dtype=CD), axes=(0, 1, 2))
        field_norm = float(jnp.sqrt(jnp.sum(jnp.abs(A_kwo)**2)))
        grad_norm = float(jnp.sqrt(gnorm2))
        max_lr = 0.01 * field_norm / (grad_norm + 1e-30)
        lr = min(lr, max_lr)

        A_kwo = A_kwo - lr * wg['grad']
        A_opt = np.asarray(jnp.fft.ifftn(A_kwo, axes=(0, 1, 2)))

        # Renormalize to fixed power
        P_cur = float(jnp.sum(jnp.abs(A_opt)**2) * dt_val)
        A_opt = A_opt * np.sqrt(P0_fixed / P_cur)

        history['step'].append(step)
        history['loss'].append(loss_val)
        history['eps'].append(eps)
        history['lr'].append(lr)

        if step % print_every == 0 or step == n_steps - 1:
            print(f"  Step {step:3d}: eps={eps:.6e}, lr={lr:.2e}, "
                  f"|grad|={grad_norm:.2e}, "
                  f"fwd={wg['fwd_seconds']:.1f}s bwd={wg['bwd_seconds']:.1f}s")

    print(f"Final eps: {history['eps'][-1]:.6e}")
    return A_opt, history

print("Loss function and optimization loop defined.")


In [ ]:
# --- N=1 Gaussian init ---
fwhm_8 = T0_8 * 2 * np.sqrt(2 * np.log(2))
E_gauss_N1 = gaussian_pulse_profile_freq(t0=0.0, fwhm=fwhm_8, Lt=Lt_8, Nt=Nt_8)
E_gauss_N1 = E_gauss_N1 * jnp.sqrt(P0_N1)
field_gauss_N1 = E_gauss_N1[None, None, :]

args_N1 = make_args(
    Nx=1, Ny=1, Nt=Nt_8,
    Lx=1e-6, Ly=1e-6, Lt=Lt_8, Lz=Lz_N1,
    n2_val=n2, beta2_val=beta2_8,
    fr=0.0, sw=0,
    deltaZ=deltaZ_8,
    save_at=[],
)

ctx_N1 = make_windowed_context(args_N1, Nt_8, precision='fp64')
_, _, replicate_8 = _make_mesh_for_time_axis(Nt_8)

print("N=1 Gaussian init optimization:")
A_opt_gauss_N1, hist_gauss_N1 = run_soliton_optimization(
    field_gauss_N1, args_N1, ctx_N1, replicate_8,
    n_steps=n_gd_steps_8, n_windows=n_windows_8)

In [ ]:
# --- N=1 Random HG init ---
rng = np.random.default_rng(42)
coeffs_rand_N1 = rng.standard_normal(n_hg_modes) + 1j * rng.standard_normal(n_hg_modes)
E_rand_N1 = np.zeros(Nt_8, dtype=np.complex128)
for k in range(n_hg_modes):
    E_rand_N1 += coeffs_rand_N1[k] * hg_basis[k]

# Scale to match Gaussian's total power: sum(|E|^2)*dt
P_gauss = float(np.sum(np.abs(np.asarray(field_gauss_N1[0, 0, :]))**2) * dt_8)
P_rand = float(np.sum(np.abs(E_rand_N1)**2) * dt_8)
E_rand_N1 = E_rand_N1 * np.sqrt(P_gauss / P_rand)
field_rand_N1 = jnp.asarray(E_rand_N1[None, None, :], dtype=jnp.complex128)

print("N=1 Random HG init optimization:")
A_opt_rand_N1, hist_rand_N1 = run_soliton_optimization(
    field_rand_N1, args_N1, ctx_N1, replicate_8,
    n_steps=n_gd_steps_8, n_windows=n_windows_8)

In [ ]:
# --- N=1 Results comparison with ground truth ---

# Ground truth: exact sech soliton at N=1 power
E_sech_N1 = np.sqrt(P0_N1) / np.cosh(t_8 / T0_8)
field_sech_N1 = jnp.asarray(E_sech_N1[None, None, :], dtype=jnp.complex128)

args_N1_viz = make_args(
    Nx=1, Ny=1, Nt=Nt_8,
    Lx=1e-6, Ly=1e-6, Lt=Lt_8, Lz=Lz_N1,
    n2_val=n2, beta2_val=beta2_8,
    fr=0.0, sw=0,
    deltaZ=deltaZ_8, n_saves=int(round(Lz_N1 / deltaZ_8)),
)

print("Propagating ground truth sech (N=1)...")
out_sech1 = propagate_windowed(args_N1_viz, field_sech_N1, ctx=ctx_N1)
print(f"  Done in {out_sech1['seconds']:.1f} s")
print("Propagating optimized (Gauss init)...")
out_g = propagate_windowed(args_N1_viz, A_opt_gauss_N1, ctx=ctx_N1)
print(f"  Done in {out_g['seconds']:.1f} s")
print("Propagating optimized (rand HG init)...")
out_r = propagate_windowed(args_N1_viz, A_opt_rand_N1, ctx=ctx_N1)
print(f"  Done in {out_r['seconds']:.1f} s")

# --- Figure: 2 rows ---
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)

# Row 1: (a) convergence, (b) pulse shapes
ax_a = fig.add_subplot(gs[0, 0])
ax_a.semilogy(hist_gauss_N1['step'], hist_gauss_N1['eps'], '-', label='Gaussian init', lw=1.5)
ax_a.semilogy(hist_rand_N1['step'], hist_rand_N1['eps'], '-', label='Random HG init', lw=1.5)
ax_a.set_xlabel('GD Step'); ax_a.set_ylabel(r'$\epsilon$ (self-consistency)')
ax_a.set_title('(a) N=1 Convergence'); ax_a.legend(); ax_a.grid(True, alpha=0.3)

ax_b = fig.add_subplot(gs[0, 1:])
I_sech_N1 = P0_N1 / np.cosh(t_8 / T0_8)**2
I_gauss_init = np.abs(np.asarray(field_gauss_N1[0, 0, :]))**2
I_gauss_opt = np.abs(A_opt_gauss_N1[0, 0, :])**2
I_rand_opt = np.abs(A_opt_rand_N1[0, 0, :])**2
ax_b.plot(t_8*1e15, I_gauss_init/I_gauss_init.max(), '-', label='Initial Gaussian', lw=2)
ax_b.plot(t_8*1e15, I_gauss_opt/I_gauss_opt.max(), '--', label='Optimized (Gauss)', lw=2)
ax_b.plot(t_8*1e15, I_rand_opt/I_rand_opt.max(), '-.', label='Optimized (rand HG)', lw=2)
ax_b.plot(t_8*1e15, I_sech_N1/I_sech_N1.max(), ':', label=r'Ground truth sech$^2$', lw=2, color='black', alpha=0.7)
ax_b.set_xlabel('t (fs)'); ax_b.set_ylabel('Normalized intensity')
ax_b.set_title('(b) N=1 Pulse Shapes'); ax_b.set_xlim([-500, 500])
ax_b.legend(); ax_b.grid(True, alpha=0.3)

# Row 2: I(t,z) heatmaps — ground truth, gauss-init optimized, rand-HG optimized
z_viz = np.asarray(out_sech1['save_at'])
extent_tz = [z_viz.min()*1e3, z_viz.max()*1e3, t_8.min()*1e15, t_8.max()*1e15]

heatmap_data = [
    (out_sech1, r'Ground truth sech (N=1)'),
    (out_g, 'Optimized (Gauss init)'),
    (out_r, 'Optimized (rand HG init)'),
]
for col, (out_i, title) in enumerate(heatmap_data):
    ax = fig.add_subplot(gs[1, col])
    F_i = np.asarray(out_i['field'])
    I_tz = np.abs(F_i[0, 0, :, :])**2
    ax.imshow(I_tz, origin='lower', aspect='auto', extent=extent_tz, cmap='hot')
    ax.set_xlabel('z (mm)'); ax.set_title(title)
    if col == 0:
        ax.set_ylabel('t (fs)')

plt.tight_layout(); plt.show()
print("Ground truth sech should show perfectly straight horizontal bands (shape-preserving).")
print("Optimized profiles should approach this pattern.")

In [ ]:
# --- N=2 Gaussian init ---
E_gauss_N2 = gaussian_pulse_profile_freq(t0=0.0, fwhm=fwhm_8, Lt=Lt_8, Nt=Nt_8)
E_gauss_N2 = E_gauss_N2 * jnp.sqrt(P0_N2)
field_gauss_N2 = E_gauss_N2[None, None, :]

args_N2 = make_args(
    Nx=1, Ny=1, Nt=Nt_8,
    Lx=1e-6, Ly=1e-6, Lt=Lt_8, Lz=Lz_N2,
    n2_val=n2, beta2_val=beta2_8,
    fr=0.0, sw=0,
    deltaZ=deltaZ_8,
    save_at=[],
)

ctx_N2 = make_windowed_context(args_N2, Nt_8, precision='fp64')

print("N=2 Gaussian init optimization:")
A_opt_gauss_N2, hist_gauss_N2 = run_soliton_optimization(
    field_gauss_N2, args_N2, ctx_N2, replicate_8,
    n_steps=n_gd_steps_8, n_windows=n_windows_8)

In [ ]:
# --- N=2 Random HG init ---
rng2 = np.random.default_rng(123)
coeffs_rand_N2 = rng2.standard_normal(n_hg_modes) + 1j * rng2.standard_normal(n_hg_modes)
E_rand_N2 = np.zeros(Nt_8, dtype=np.complex128)
for k in range(n_hg_modes):
    E_rand_N2 += coeffs_rand_N2[k] * hg_basis[k]

# Scale to match N=2 Gaussian's total power
P_gauss_N2 = float(np.sum(np.abs(np.asarray(field_gauss_N2[0, 0, :]))**2) * dt_8)
P_rand_N2 = float(np.sum(np.abs(E_rand_N2)**2) * dt_8)
E_rand_N2 = E_rand_N2 * np.sqrt(P_gauss_N2 / P_rand_N2)
field_rand_N2 = jnp.asarray(E_rand_N2[None, None, :], dtype=jnp.complex128)

print("N=2 Random HG init optimization:")
A_opt_rand_N2, hist_rand_N2 = run_soliton_optimization(
    field_rand_N2, args_N2, ctx_N2, replicate_8,
    n_steps=n_gd_steps_8, n_windows=n_windows_8)

In [ ]:
# --- N=2 Results comparison with ground truth ---

# Ground truth: exact sech soliton at N=2 power
E_sech_N2 = np.sqrt(P0_N2) / np.cosh(t_8 / T0_8)
field_sech_N2 = jnp.asarray(E_sech_N2[None, None, :], dtype=jnp.complex128)

args_N2_viz = make_args(
    Nx=1, Ny=1, Nt=Nt_8,
    Lx=1e-6, Ly=1e-6, Lt=Lt_8, Lz=Lz_N2,
    n2_val=n2, beta2_val=beta2_8,
    fr=0.0, sw=0,
    deltaZ=deltaZ_8, n_saves=int(round(Lz_N2 / deltaZ_8)),
)

print("Propagating ground truth sech (N=2)...")
out_sech2 = propagate_windowed(args_N2_viz, field_sech_N2, ctx=ctx_N2)
print(f"  Done in {out_sech2['seconds']:.1f} s")
print("Propagating optimized N=2 (Gauss init)...")
out_g2 = propagate_windowed(args_N2_viz, A_opt_gauss_N2, ctx=ctx_N2)
print(f"  Done in {out_g2['seconds']:.1f} s")
print("Propagating optimized N=2 (rand HG init)...")
out_r2 = propagate_windowed(args_N2_viz, A_opt_rand_N2, ctx=ctx_N2)
print(f"  Done in {out_r2['seconds']:.1f} s")

# --- Figure: 2 rows ---
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)

# Row 1: (a) convergence, (b) pulse shapes
ax_a = fig.add_subplot(gs[0, 0])
ax_a.semilogy(hist_gauss_N2['step'], hist_gauss_N2['eps'], '-', label='Gaussian init', lw=1.5)
ax_a.semilogy(hist_rand_N2['step'], hist_rand_N2['eps'], '-', label='Random HG init', lw=1.5)
ax_a.set_xlabel('GD Step'); ax_a.set_ylabel(r'$\epsilon$ (self-consistency)')
ax_a.set_title('(a) N=2 Convergence'); ax_a.legend(); ax_a.grid(True, alpha=0.3)

ax_b = fig.add_subplot(gs[0, 1:])
I_sech_N2 = P0_N2 / np.cosh(t_8 / T0_8)**2
I_gauss_init_N2 = np.abs(np.asarray(field_gauss_N2[0, 0, :]))**2
I_gauss_opt_N2 = np.abs(A_opt_gauss_N2[0, 0, :])**2
I_rand_opt_N2 = np.abs(A_opt_rand_N2[0, 0, :])**2
ax_b.plot(t_8*1e15, I_gauss_init_N2/I_gauss_init_N2.max(), '-', label='Initial Gaussian', lw=2)
ax_b.plot(t_8*1e15, I_gauss_opt_N2/I_gauss_opt_N2.max(), '--', label='Optimized (Gauss)', lw=2)
ax_b.plot(t_8*1e15, I_rand_opt_N2/I_rand_opt_N2.max(), '-.', label='Optimized (rand HG)', lw=2)
ax_b.plot(t_8*1e15, I_sech_N2/I_sech_N2.max(), ':', label=r'Ground truth sech$^2$', lw=2, color='black', alpha=0.7)
ax_b.set_xlabel('t (fs)'); ax_b.set_ylabel('Normalized intensity')
ax_b.set_title('(b) N=2 Pulse Shapes'); ax_b.set_xlim([-500, 500])
ax_b.legend(); ax_b.grid(True, alpha=0.3)

# Row 2: I(t,z) heatmaps — ground truth, gauss-init optimized, rand-HG optimized
z_viz2 = np.asarray(out_sech2['save_at'])
extent_tz2 = [z_viz2.min()*1e3, z_viz2.max()*1e3, t_8.min()*1e15, t_8.max()*1e15]

heatmap_data = [
    (out_sech2, r'Ground truth sech (N=2)'),
    (out_g2, 'Optimized (Gauss init)'),
    (out_r2, 'Optimized (rand HG init)'),
]
for col, (out_i, title) in enumerate(heatmap_data):
    ax = fig.add_subplot(gs[1, col])
    F_i = np.asarray(out_i['field'])
    I_tz = np.abs(F_i[0, 0, :, :])**2
    ax.imshow(I_tz, origin='lower', aspect='auto', extent=extent_tz2, cmap='hot')
    ax.set_xlabel('z (mm)'); ax.set_title(title)
    if col == 0:
        ax.set_ylabel('t (fs)')

plt.tight_layout(); plt.show()
print("Ground truth sech (N=2) should show periodic breathing with shape return at z = z_sp.")
print("Optimized profiles should approach this breathing pattern.")